In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 20)
plt.rcParams["figure.dpi"] = 110

In [49]:
# Natural Earth 110m land polygons — used for all maps
WORLD_URL = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector"
             "/master/geojson/ne_110m_land.geojson")
world = gpd.read_file(WORLD_URL)
print("World land polygons loaded:", world.shape)

World land polygons loaded: (127, 4)


In [50]:
# ── Load IBTrACS ──────────────────────────────────────────────────────────
cols = ["SID", "SEASON", "NAME", "BASIN", "LAT", "LON",
        "USA_WIND", "USA_SSHS", "ISO_TIME"]

ibt = pd.read_csv(
    "data/ibtracs.since1980.list.v04r01.csv",
    skiprows=[1],           # skip the unit row
    usecols=cols,
    low_memory=False,
    keep_default_na=False,  # prevent "NA" (North Atlantic) being read as NaN
    na_values=[""]
)

# Convert types
for c in ["USA_WIND", "LAT", "LON", "SEASON"]:
    ibt[c] = pd.to_numeric(ibt[c], errors="coerce")

# Harmonise period: keep >= 1980
ibt = ibt[ibt["SEASON"] >= 1980].copy()

print(f"IBTrACS loaded: {len(ibt):,} position fixes, {ibt['SID'].nunique():,} unique storms")
ibt.head(3)

IBTrACS loaded: 306,064 position fixes, 4,932 unique storms


,SID,SEASON,BASIN,NAME,ISO_TIME,LAT,LON,USA_WIND,USA_SSHS
0,1980001S13173,1980,SP,PENI,1980-01-01 00:00:00,-12.5,172.5,25.0,-1
1,1980001S13173,1980,SP,PENI,1980-01-01 03:00:00,-12.2,172.4,25.0,-1
2,1980001S13173,1980,SP,PENI,1980-01-01 06:00:00,-11.9,172.4,25.0,-1


In [51]:
# ── Preprocess IBTrACS ─────────────────────────────────────────────────────
ibt_clean = ibt.dropna(subset=["USA_WIND", "LAT", "LON", "SEASON"]).copy()

# --- storms: global peak per storm (used for maps and counting) ---
storms = (ibt_clean
          .loc[ibt_clean.groupby("SID")["USA_WIND"].idxmax()]
          .drop_duplicates(subset=["SID"])
          .copy())
storms["SEASON"] = storms["SEASON"].astype(int)

# NAME_NORM = NAME_YEAR_MONTH  e.g. KATRINA_2005_08
_ym_s = pd.to_datetime(storms["ISO_TIME"], errors="coerce").dt.strftime("%Y_%m")
storms["NAME_NORM"] = storms["NAME"].str.strip().str.upper() + "_" + _ym_s

# --- storms_land: near-land peak per storm (used for mortality analysis) ---
world_buf = gpd.GeoDataFrame(geometry=world.geometry.buffer(1.0), crs="EPSG:4326")
ibt_geo   = gpd.GeoDataFrame(
    ibt_clean,
    geometry=gpd.points_from_xy(ibt_clean["LON"], ibt_clean["LAT"]),
    crs="EPSG:4326"
)
near        = gpd.sjoin(ibt_geo, world_buf[["geometry"]], how="inner", predicate="within")
near        = near[~near.index.duplicated(keep="first")]
storms_land = (near
               .sort_values("USA_WIND", ascending=False)
               .drop_duplicates(subset=["SID"])
               .copy())
storms_land["SEASON"] = storms_land["SEASON"].astype(int)

# Same NAME_NORM format: NAME_YEAR_MONTH  e.g. KATRINA_2005_08
_ym_l = pd.to_datetime(storms_land["ISO_TIME"], errors="coerce").dt.strftime("%Y_%m")
storms_land["NAME_NORM"] = storms_land["NAME"].str.strip().str.upper() + "_" + _ym_l

print(f"Total storms (global peak):  {len(storms):,}")
print(f"Storms with near-land track: {len(storms_land):,}")
print(f"Period: {storms['SEASON'].min()}–{storms['SEASON'].max()}")
print(f"Basin breakdown:")
print(storms["BASIN"].value_counts().to_frame("storms"))

# Expand basin codes to full names
convert_basins = {
    "NA": "North Atlantic",
    "WP": "Western Pacific",
    "EP": "Eastern Pacific",
    "NI": "North Indian",
    "SI": "South Indian",
    "SP": "South Pacific",
    "SA": "South Atlantic",
}
storms["BASIN"]      = storms["BASIN"].replace(convert_basins)
storms_land["BASIN"] = storms_land["BASIN"].replace(convert_basins)

# Show sample
print("Sample NAME_NORM values:")
print(storms_land[["NAME", "SEASON", "NAME_NORM"]].head(5).to_string(index=False))

storms.to_csv("data_final/ibtracs_storms.csv", index=False)
storms_land.to_csv("data_final/ibtracs_storms_land.csv", index=False)

Total storms (global peak):  4,488
Storms with near-land track: 2,309
Period: 1980–2026
Basin breakdown:
       storms
BASIN        
WP       1372
EP        905
SI        768
NA        741
SP        456
NI        243
SA          3
Sample NAME_NORM values:
   NAME  SEASON       NAME_NORM
 HAIYAN    2013  HAIYAN_2013_11
MELISSA    2025 MELISSA_2025_10
  ALLEN    1980   ALLEN_1980_08
 DORIAN    2019  DORIAN_2019_09
 MONICA    2006  MONICA_2006_04


In [52]:
# ── Load EM-DAT ────────────────────────────────────────────────────────────
em = pd.read_excel("data/emdat_tropical_cyclones.xlsx")


# Numeric conversions
for col in ["Total Deaths", "No. Injured", "Total Affected",
            "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)"]:
    em[col] = pd.to_numeric(em[col], errors="coerce")

# Keep relevant columns (user selection)
cols_to_keep = [
    "ISO", "Country", "Subregion", "Region",
    "Start Year", "Start Month", "Start Day",
    "End Year",   "End Month",   "End Day",
    "Total Deaths", "No. Injured", "Total Affected",
    "Total Damage ('000 US$)", "Total Damage, Adjusted ('000 US$)", 
    "Event Name"
]
em = em[cols_to_keep].copy()
em = em.dropna(subset=["Start Month"])  # drop events with missing start year

print(f"EM-DAT loaded: {len(em):,} events (post-1980)")
em.head(3)
em = em[em["Start Year"] >= 1980].copy()


EM-DAT loaded: 2,680 events (post-1980)


In [ ]:
# ── Clean Event Name ─────────────────────────────────────────────────────────
_PREFIXES = (
    r"^(HURRICANE|TYPHOON|CYCLONE|TROPICAL STORM|TROPICAL CYCLONE|"
    r"SUPER TYPHOON|SUPER CYCLONE|SEVERE TROPICAL STORM)\s+"
)
em["Event Name"] = (em["Event Name"]
    .str.upper()
    .str.replace("'", "", regex=False)            # remove single quotes (e.g. Cyclone 'Nargis')
    .str.replace(_PREFIXES, "", regex=True)        # strip leading type prefix
    .str.replace(r"\([^)]*\)", "", regex=True)   # remove (parenthetical text)
    .str.replace(r"[^A-Z0-9 ]", " ", regex=True)  # keep only alphanum + spaces
    .str.replace(r"\s+", " ", regex=True)        # collapse spaces
    .str.strip())

print(f"Event Names non-null: {em['Event Name'].notna().sum()} / {len(em)}")
print("Sample:", em["Event Name"].dropna().head(5).tolist())

# ── Build NAME_NORM for EM-DAT ───────────────────────────────────────────────
# For each event: split the cleaned Event Name into words, then check if any
# word matches an IBTrACS storm name (whole-word, uppercase).
# This avoids large-alternation regex issues.
# Result format: {IBTrACS_NAME}_{YEAR}_{MONTH:02d}   e.g. KATRINA_2005_08

_SKIP = {"NOT_NAMED", "UNNAMED", "NO-NAME", "NONAME", "INVEST"}
ibt_name_set = {
    n.strip().upper()
    for n in storms_land["NAME"].unique()
    if isinstance(n, str) and len(n.strip()) > 2 and n.strip().upper() not in _SKIP
}
print(f"IBTrACS names available for matching: {len(ibt_name_set)}")

def _name_norm(row):
    event = str(row["Event Name"]) if pd.notna(row["Event Name"]) else ""
    if not event:
        return None
    words  = set(event.split())
    match  = words & ibt_name_set      # whole-word intersection (O(n_words))
    if not match:
        return None
    name  = max(match, key=len)        # longest match wins if multiple
    year  = int(row["Start Year"])
    month = int(row["Start Month"])
    return f"{name}_{year}_{month:02d}"

em["NAME_NORM"] = em.apply(_name_norm, axis=1)

n_matched = em["NAME_NORM"].notna().sum()
print(f"Matched: {n_matched:,} / {len(em):,} ({n_matched/len(em)*100:.1f}%)")
print("Sample:")
print(em[["Event Name", "Country", "Start Year", "Start Month", "NAME_NORM"]]
      .dropna(subset=["NAME_NORM"]).head(10).to_string(index=False))

Event Names non-null: 1985 / 2124
Sample: ['HYACINTHE', 'WALLY', 'JOE', 'JOE', 'JOE']
IBTrACS names available for matching: 1075
Matched: 1,704 / 2,124 (80.2%)
Sample:
Event Name                          Country  Start Year  Start Month         NAME_NORM
 HYACINTHE                          Réunion        1980          1.0 HYACINTHE_1980_01
     WALLY                             Fiji        1980          3.0     WALLY_1980_03
       JOE                      Philippines        1980          7.0       JOE_1980_07
       JOE                         Viet Nam        1980          7.0       JOE_1980_07
       JOE                            China        1980          7.0       JOE_1980_07
       KIM                      Philippines        1980          7.0       KIM_1980_07
     ALLEN                      Saint Lucia        1980          7.0     ALLEN_1980_07
     ALLEN Saint Vincent and the Grenadines        1980          7.0     ALLEN_1980_07
     ALLEN                            Haiti      

In [ ]:
em.to_csv("data_final/emdat_tropical_cyclones_preprocessed.csv", index=False)